In [0]:
dbutils.widgets.text("catalog", "dev")
catalog = dbutils.widgets.get("catalog")

from pyspark.sql import functions as F
from pyspark.sql.window import Window

DATE_START = "1992-01-01"
DATE_END = "1998-12-31"

In [0]:
def validate_primary_key(df, pk_columns, table_name):
    pk_label = ", ".join(pk_columns)

    # 1. Null check: True if ANY pk column is null
    has_null = F.lit(False)
    for col in pk_columns:
        has_null = has_null | F.col(col).isNull()

    # 2. Duplicate check: count rows sharing the same PK combination
    #    If count > 1, this row is a duplicate
    window = Window.partitionBy(*pk_columns)
    is_duplicate = F.count(F.lit(1)).over(window) > 1

    # 3. Add both check columns at once (withColumns avoids nested plan)
    df = df.withColumns({
        "_check_pk_null": F.when(has_null, f"PK_NULL: [{pk_label}] contains null values")
                          .otherwise(F.lit(None).cast("string")),
        "_check_pk_dup": F.when(is_duplicate, f"PK_DUPLICATE: [{pk_label}] has duplicate values")
                         .otherwise(F.lit(None).cast("string")),
    })

    metric_specs = [
        ("_check_pk_null", "Primary Key", f"Null check on [{pk_label}] in {table_name}"),
        ("_check_pk_dup", "Primary Key", f"Duplicate check on [{pk_label}] in {table_name}"),
    ]
    return df, metric_specs

In [0]:
def validate_numeric_range(df, range_rules, table_name):
    metric_specs = []
    new_cols = {}

    for rule in range_rules:
        col_name = rule["column"]
        min_val  = rule.get("min_val")
        max_val  = rule.get("max_val")
        min_excl = rule.get("min_exclusive", False)
        max_excl = rule.get("max_exclusive", False)
        label    = rule.get("label", col_name)

        # Build violation condition:
        # A row violates if the value is null, below min, or above max
        is_violation = F.col(col_name).isNull()

        if min_val is not None:
            if min_excl:
                is_violation = is_violation | (F.col(col_name) <= min_val)
            else:
                is_violation = is_violation | (F.col(col_name) < min_val)

        if max_val is not None:
            if max_excl:
                is_violation = is_violation | (F.col(col_name) >= max_val)
            else:
                is_violation = is_violation | (F.col(col_name) > max_val)

        # Build a readable description of the valid range
        parts = []
        if min_val is not None:
            parts.append(f"{'>' if min_excl else '>='} {min_val}")
        if max_val is not None:
            parts.append(f"{'<' if max_excl else '<='} {max_val}")
        range_desc = " and ".join(parts) if parts else "not null"

        reason = f"NUMERIC_RANGE: {col_name} must be {range_desc}"
        check_col = f"_check_numeric_{label}"
        new_cols[check_col] = F.when(is_violation, reason).otherwise(F.lit(None).cast("string"))

        metric_specs.append((check_col, "Numeric Range", f"{col_name} must be {range_desc} in {table_name}"))

    # Add all check columns at once (avoids nested plan from withColumn in loop)
    df = df.withColumns(new_cols)
    return df, metric_specs

In [0]:
def validate_date_range(df, date_columns, start_date, end_date, table_name):
    metric_specs = []
    new_cols = {}

    for col_name in date_columns:
        # A row violates if the date is null, before start, or after end
        is_violation = (
            F.col(col_name).isNull()
            | (F.col(col_name) < F.lit(start_date).cast("date"))
            | (F.col(col_name) > F.lit(end_date).cast("date"))
        )

        reason = f"DATE_RANGE: {col_name} must be between {start_date} and {end_date}"
        check_col = f"_check_date_{col_name}"
        new_cols[check_col] = F.when(is_violation, reason).otherwise(F.lit(None).cast("string"))

        metric_specs.append((check_col, "Date Range", f"{col_name} in [{start_date}, {end_date}] in {table_name}"))

    # Add all check columns at once
    df = df.withColumns(new_cols)
    return df, metric_specs

In [0]:
def validate_referential_integrity(df, fk_column, parent_df, parent_pk_column, table_name):
    # Get the set of valid parent keys (Silver parent already passed PK validation)
    parent_keys = parent_df.select(F.col(parent_pk_column).alias("_parent_key")).distinct()

    # Left join: if _parent_key is null after join, the FK had no match
    df = df.join(parent_keys, df[fk_column] == parent_keys["_parent_key"], "left")

    check_col = f"_check_ri_{fk_column}"
    df = df.withColumns({
        check_col: F.when(F.col(fk_column).isNull(), f"RI_NULL: Foreign key [{fk_column}] is null")
                    .when(F.col("_parent_key").isNull(), f"RI_ORPHAN: [{fk_column}] has no matching parent in [{parent_pk_column}]")
                    .otherwise(F.lit(None).cast("string"))
    }).drop("_parent_key")

    metric_specs = [(check_col, "Referential Integrity", f"{fk_column} -> {parent_pk_column} in {table_name}")]
    return df, metric_specs

In [0]:
def clean_table(table_name, catalog, pk_columns,
                numeric_rules=None, date_columns=None,
                ri_relationships=None, dq_metrics_list=None):

    # --- Table names ---
    bronze_table    = f"{catalog}.bronze.{table_name}"
    silver_table    = f"{catalog}.silver.{table_name}"
    quarantine_tbl  = f"{catalog}.silver.{table_name}_quarantine"
    temp_table      = f"{catalog}.silver._tmp_clean_{table_name}"

    # --- Step 1: Read Bronze, count rows, drop metadata columns ---
    bronze_df = spark.read.table(bronze_table)
    bronze_count = bronze_df.count()
    print(f"[{table_name}] Bronze: {bronze_count:,}")

    # Drop Bronze metadata columns — not needed in Silver
    df = bronze_df.drop("file_path", "ingestion_time")
    all_metric_specs = []

    # --- Step 2: Run validations ---

    # 2a. Primary key: null + duplicate check
    df, specs = validate_primary_key(df, pk_columns, table_name)
    all_metric_specs.extend(specs)

    # 2b. Numeric range (e.g., quantity > 0, discount 0-10%)
    if numeric_rules:
        df, specs = validate_numeric_range(df, numeric_rules, table_name)
        all_metric_specs.extend(specs)

    # 2c. Date range (must be within 1992-1998)
    if date_columns:
        df, specs = validate_date_range(df, date_columns, DATE_START, DATE_END, table_name)
        all_metric_specs.extend(specs)

    # 2d. Referential integrity (FK must exist in Silver parent table)
    if ri_relationships:
        for ri in ri_relationships:
            parent_df = spark.read.table(f"{catalog}.silver.{ri['parent_table']}")
            df, specs = validate_referential_integrity(
                df, ri["fk_column"], parent_df, ri["parent_pk_column"], table_name
            )
            all_metric_specs.extend(specs)

    # --- Step 3: Combine all _check_ columns into one _rejection_reasons column ---
    # Each validation added a _check_* column. Null means "passed".
    # We join all non-null reasons with " | " to get the full rejection reason.
    check_cols = [spec[0] for spec in all_metric_specs]
    combined_reasons = F.concat_ws(" | ", *[F.col(c) for c in check_cols])

    df = df.withColumns({
        "_rejection_reasons": F.when(combined_reasons == "", F.lit(None)).otherwise(combined_reasons)
    })

    # --- Step 4: Materialize to temp table ---
    # We need the data 3 more times (metrics + write Silver + write Quarantine).
    # .cache() is not supported on serverless, so we write to a temp table instead.
    df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(temp_table)
    df = spark.read.table(temp_table)

    # --- Step 5: Compute metrics in one pass ---
    # Count valid records, quarantined records, and per-rule failures
    agg_exprs = [
        F.sum(F.when(F.col("_rejection_reasons").isNull(), 1).otherwise(0)).alias("_silver_count"),
        F.sum(F.when(F.col("_rejection_reasons").isNotNull(), 1).otherwise(0)).alias("_quarantine_count"),
    ]
    for check_col, _, _ in all_metric_specs:
        agg_exprs.append(
            F.sum(F.when(F.col(check_col).isNotNull(), 1).otherwise(0)).alias(check_col)
        )

    metrics = df.agg(*agg_exprs).collect()[0]
    silver_count    = metrics["_silver_count"]
    quarantine_cnt  = metrics["_quarantine_count"]

    # Record per-rule metrics for the DQ report
    for check_col, category, rule_desc in all_metric_specs:
        fail_count = metrics[check_col]
        if dq_metrics_list is not None:
            dq_metrics_list.append({
                "table": table_name,
                "rule_category": category,
                "rule": rule_desc,
                "failing_records": fail_count,
                "action": "Quarantined" if fail_count > 0 else "None - all records passed",
            })

    # --- Step 6: Split and write ---
    # Silver table: clean records only, no check columns
    valid_df = df.filter(F.col("_rejection_reasons").isNull()) \
                 .drop(*check_cols, "_rejection_reasons")

    # Quarantine table: rejected records, keep _rejection_reasons for audit
    quarantine_df = df.filter(F.col("_rejection_reasons").isNotNull()) \
                      .drop(*check_cols)

    valid_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(silver_table)
    quarantine_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(quarantine_tbl)

    # Clean up temp table
    spark.sql(f"DROP TABLE IF EXISTS {temp_table}")

    print(f"[{table_name}] Silver: {silver_count:,} | Quarantine: {quarantine_cnt:,}")

    return {
        "table": table_name,
        "bronze_count": bronze_count,
        "silver_count": silver_count,
        "quarantine_count": quarantine_cnt,
    }

In [0]:
def generate_dq_report(dq_metrics):
    if not dq_metrics:
        print("No data-quality metrics to report.")
        return None

    report_df = spark.createDataFrame(dq_metrics)
    report_df = report_df.orderBy("table", "rule_category", "rule")
    print("=== Data Quality Report ===")
    display(report_df)
    report_df.write.mode("overwrite").saveAsTable(f"{catalog}.silver.data_quality_report")
    return report_df

In [0]:
# Configuration for all eight TPC-H tables
# Each table specifies which validations to apply:
#   pk_columns       -- list of PK column names (single or composite)
#   numeric_rules     -- list of range-rule dicts (lineitem only)
#   date_columns      -- list of date columns to validate (orders, lineitem)
#   ri_relationships  -- list of FK -> parent mappings

TABLE_CONFIGS = {
    # Geography hierarchy: region -> nation
    "region": {
        "pk_columns": ["r_regionkey"],
    },
    "nation": {
        "pk_columns": ["n_nationkey"],
        "ri_relationships": [
            {"fk_column": "n_regionkey", "parent_table": "region", "parent_pk_column": "r_regionkey"},
        ],
    },
    # Product catalog
    "part": {
        "pk_columns": ["p_partkey"],
    },
    # Supplier -> nation, then partsupp -> part + supplier
    "supplier": {
        "pk_columns": ["s_suppkey"],
        "ri_relationships": [
            {"fk_column": "s_nationkey", "parent_table": "nation", "parent_pk_column": "n_nationkey"},
        ],
    },
    "customer": {
        "pk_columns": ["c_custkey"],
        "ri_relationships": [
            {"fk_column": "c_nationkey", "parent_table": "nation", "parent_pk_column": "n_nationkey"},
        ],
    },
    "partsupp": {
        "pk_columns": ["ps_partkey", "ps_suppkey"],  # composite PK
        "ri_relationships": [
            {"fk_column": "ps_partkey", "parent_table": "part", "parent_pk_column": "p_partkey"},
            {"fk_column": "ps_suppkey", "parent_table": "supplier", "parent_pk_column": "s_suppkey"},
        ],
    },
    # Order lifecycle: orders -> customer, lineitem -> orders + part + supplier
    "orders": {
        "pk_columns": ["o_orderkey"],
        "date_columns": ["o_orderdate"],
        "ri_relationships": [
            {"fk_column": "o_custkey", "parent_table": "customer", "parent_pk_column": "c_custkey"},
        ],
    },
    "lineitem": {
        "pk_columns": ["l_orderkey", "l_linenumber"],  # composite PK
        "numeric_rules": [
            {"column": "l_quantity", "min_val": 0, "min_exclusive": True, "label": "l_quantity"},
            {"column": "l_extendedprice", "min_val": 0, "min_exclusive": True, "label": "l_extendedprice"},
            {"column": "l_discount", "min_val": 0, "max_val": 0.10, "min_exclusive": False, "max_exclusive": False, "label": "l_discount"},
            {"column": "l_tax", "min_val": 0, "max_val": 0.08, "min_exclusive": False, "max_exclusive": False, "label": "l_tax"},
        ],
        "date_columns": ["l_shipdate", "l_commitdate", "l_receiptdate"],
        "ri_relationships": [
            {"fk_column": "l_orderkey", "parent_table": "orders", "parent_pk_column": "o_orderkey"},
            {"fk_column": "l_partkey", "parent_table": "part", "parent_pk_column": "p_partkey"},
            {"fk_column": "l_suppkey", "parent_table": "supplier", "parent_pk_column": "s_suppkey"},
        ],
    },
}

# Processing order ensures parent Silver tables exist before child RI checks
# region -> nation -> part -> supplier -> customer -> partsupp -> orders -> lineitem
PROCESSING_ORDER = ["region", "nation", "part", "supplier", "customer", "partsupp", "orders", "lineitem"]

In [0]:
dq_metrics = []

for table_name in PROCESSING_ORDER:
    config = TABLE_CONFIGS[table_name]
    clean_table(
        table_name=table_name,
        catalog=catalog,
        pk_columns=config["pk_columns"],
        numeric_rules=config.get("numeric_rules"),
        date_columns=config.get("date_columns"),
        ri_relationships=config.get("ri_relationships"),
        dq_metrics_list=dq_metrics,
    )

print("\nAll tables processed.")

In [0]:
generate_dq_report(dq_metrics)